# EstateIQ AI — Visual Exploratory Data Analysis (EDA)

**Objective:** Uncover distribution characteristics, spatial price variance, feature interactions, and domain anomalies across metropolitan Bengaluru residential real estate.

---

## 1. Environment Setup & Data Ingestion

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting settings
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid", palette="muted")

# Ensure figures directory exists
# If running inside notebooks/, root is '..'
# If running from project root, root is '.'
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
figures_dir = project_root / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

data_path = project_root / "data" / "raw" / "Bengaluru_House_Data.csv"
df_raw = pd.read_csv(data_path)

print(f"Dataset successfully loaded from: {data_path.resolve()}")
print(f"Shape: {df_raw.shape[0]:,} listings across {df_raw.shape[1]} features")
df_raw.head(3)

---

## 2. Target Variable Analysis: Raw Price vs. Log-Transformed Price

In real estate valuation, property prices almost always exhibit a heavy right-skewed distribution. 
Here we compare the raw asking price ($y$) against the natural logarithm $\log(y)$ to evaluate normality.

In [ ]:
prices = df_raw["price"]
log_prices = np.log(prices)

raw_skew = prices.skew()
log_skew = log_prices.skew()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Raw Price Distribution with Mean and Median markers
sns.histplot(prices, kde=True, color="#1f77b4", ax=axes[0], bins=50)
axes[0].axvline(prices.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean: ₹{prices.mean():.1f} L")
axes[0].axvline(prices.median(), color="green", linestyle="-", linewidth=2, label=f"Median: ₹{prices.median():.1f} L")
axes[0].set_title(f"Raw Price Distribution (Skewness: {raw_skew:.2f})", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Price in Lakhs (₹)")
axes[0].set_ylabel("Count (Frequency)")
axes[0].set_xlim(0, 1000)  # Zoom in on main density for visual clarity
axes[0].legend(frameon=True)

# Subplot 2: Log-Transformed Price Distribution
sns.histplot(log_prices, kde=True, color="#2ca02c", ax=axes[1], bins=40)
axes[1].axvline(log_prices.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean: {log_prices.mean():.2f}")
axes[1].axvline(log_prices.median(), color="green", linestyle="-", linewidth=2, label=f"Median: {log_prices.median():.2f}")
axes[1].set_title(f"Log-Transformed Price Distribution (Skewness: {log_skew:.2f})", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Log(Price in Lakhs)")
axes[1].set_ylabel("Count (Frequency)")
axes[1].legend(frameon=True)

plt.suptitle("EstateIQ AI — Target Variable (Price) Distribution Diagnostics", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

# Save figure to reports/figures/
save_path = figures_dir / "01_price_distribution.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved to: {save_path.resolve()}")
print(f"Raw Price Skewness        : {raw_skew:.2f} (Extremely Right-Skewed)")
print(f"Log-Transformed Skewness  : {log_skew:.2f} (Near-Symmetric Gaussian Distribution)")